In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


In [2]:
# ============================================================
# 1. LOAD DATASET
# ============================================================

train = pd.read_csv("en_ta_small_clean_test_dataset.csv")

# Support both column names used in your files:
#   english / tamil
#   en / ta

if "english" in train.columns and "tamil" in train.columns:
    english_sentences = train["english"].astype(str).tolist()
    tamil_sentences = train["tamil"].astype(str).tolist()
elif "en" in train.columns and "ta" in train.columns:
    english_sentences = train["en"].astype(str).tolist()
    tamil_sentences = train["ta"].astype(str).tolist()
else:
    raise ValueError(
        f"Expected columns ['english','tamil'] or ['en','ta']. Found: {list(train.columns)}"
    )

# Remove empty rows and duplicates
pairs = pd.DataFrame({
    "english": english_sentences,
    "tamil": tamil_sentences
})

pairs = pairs.dropna()
pairs = pairs[
    (pairs["english"].str.strip() != "") &
    (pairs["tamil"].str.strip() != "")
]
pairs = pairs.drop_duplicates().reset_index(drop=True)

english_sentences = pairs["english"].tolist()
tamil_sentences = pairs["tamil"].tolist()

print("Total pairs:", len(pairs))
print()
print("Example:")
print("EN:", english_sentences[0])
print("TA:", tamil_sentences[0])


Total pairs: 101

Example:
EN: How are you?
TA: நீங்கள் எப்படி இருக்கிறீர்கள்?


In [3]:
# ============================================================
# 2. ADD SOS / EOS TO TAMIL TARGETS
# ============================================================

SOS = "<sos>"
EOS = "<eos>"

tamil_sent_SE = [
    f"{SOS} {sentence} {EOS}"
    for sentence in tamil_sentences
]

print("English:")
print(english_sentences[0])

print("\nTamil target:")
print(tamil_sent_SE[0])


English:
How are you?

Tamil target:
<sos> நீங்கள் எப்படி இருக்கிறீர்கள்? <eos>


In [4]:
# ============================================================
# 3. TOKENIZERS
# ============================================================

english_tokenizer = Tokenizer(
    filters="",
    oov_token="<unk>"
)

tamil_tokenizer = Tokenizer(
    filters="",
    oov_token="<unk>"
)

english_tokenizer.fit_on_texts(english_sentences)
tamil_tokenizer.fit_on_texts(tamil_sent_SE)

english_sequences = english_tokenizer.texts_to_sequences(
    english_sentences
)

tamil_sequences = tamil_tokenizer.texts_to_sequences(
    tamil_sent_SE
)

english_vocab_size = len(english_tokenizer.word_index) + 1
tamil_vocab_size = len(tamil_tokenizer.word_index) + 1

print("English vocabulary:", english_vocab_size)
print("Tamil vocabulary:", tamil_vocab_size)


English vocabulary: 172
Tamil vocabulary: 179


In [5]:
# ============================================================
# 4. MAXIMUM SEQUENCE LENGTHS
# ============================================================

max_input_seq_length = max(
    len(x) for x in english_sequences
)

max_output_seq_length = max(
    len(x) for x in tamil_sequences
)

print("Max English length:", max_input_seq_length)
print("Max Tamil length:", max_output_seq_length)


Max English length: 8
Max Tamil length: 7


In [6]:
# ============================================================
# 5. PADDING
# ============================================================

input_sequences = pad_sequences(
    english_sequences,
    maxlen=max_input_seq_length,
    padding="post"
)

output_sequences = pad_sequences(
    tamil_sequences,
    maxlen=max_output_seq_length,
    padding="post"
)

print("Input shape:", input_sequences.shape)
print("Output shape:", output_sequences.shape)


Input shape: (101, 8)
Output shape: (101, 7)


In [7]:
# ============================================================
# 6. DECODER INPUT / TARGET
# ============================================================

decoder_input_sequences = output_sequences[:, :-1]
decoder_target_ids = output_sequences[:, 1:]

print("Decoder input shape:", decoder_input_sequences.shape)
print("Decoder target shape:", decoder_target_ids.shape)


Decoder input shape: (101, 6)
Decoder target shape: (101, 6)


In [8]:
# ============================================================
# 7. ONE-HOT TARGETS + PADDING MASK
# ============================================================

decoder_output_sequences = tf.keras.utils.to_categorical(
    decoder_target_ids,
    num_classes=tamil_vocab_size
)

# Padding token ID is 0.
# Do not allow padded positions to contribute to the loss/accuracy.
sample_weight = (decoder_target_ids != 0).astype("float32")

print("Decoder output shape:", decoder_output_sequences.shape)
print("Sample-weight shape:", sample_weight.shape)


Decoder output shape: (101, 6, 179)
Sample-weight shape: (101, 6)


In [9]:
# ============================================================
# 8. VERIFY ONE EXAMPLE
# ============================================================

tamil_index_word = {
    index: word
    for word, index in tamil_tokenizer.word_index.items()
}

i = 0

print("English:")
print(english_sentences[i])

print("\nFull Tamil:")
print(tamil_sent_SE[i])

print("\nDecoder input:")
print([
    tamil_index_word.get(int(x), "<UNK>")
    for x in decoder_input_sequences[i]
    if x != 0
])

print("\nDecoder target:")
print([
    tamil_index_word.get(int(x), "<UNK>")
    for x in decoder_target_ids[i]
    if x != 0
])


English:
How are you?

Full Tamil:
<sos> நீங்கள் எப்படி இருக்கிறீர்கள்? <eos>

Decoder input:
['<sos>', 'நீங்கள்', 'எப்படி', 'இருக்கிறீர்கள்?', '<eos>']

Decoder target:
['நீங்கள்', 'எப்படி', 'இருக்கிறீர்கள்?', '<eos>']


In [10]:
# ============================================================
# 9. ENCODER
# ============================================================

EMBED_DIM = 64
LSTM_UNITS = 128

encoder_input = Input(
    shape=(max_input_seq_length,),
    name="encoder_input"
)

encoder_embedding = Embedding(
    input_dim=english_vocab_size,
    output_dim=EMBED_DIM,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_input)

encoder_lstm = LSTM(
    LSTM_UNITS,
    return_state=True,
    name="encoder_lstm"
)

_, state_h, state_c = encoder_lstm(
    encoder_embedding
)


In [11]:
# ============================================================
# 10. DECODER
# ============================================================

decoder_input_layer = Input(
    shape=(max_output_seq_length - 1,),
    name="decoder_input"
)

decoder_embedding_layer = Embedding(
    input_dim=tamil_vocab_size,
    output_dim=EMBED_DIM,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(
    decoder_input_layer
)

decoder_lstm = LSTM(
    LSTM_UNITS,
    return_sequences=True,
    return_state=True,
    name="decoder_lstm"
)

decoder_output, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

decoder_dense = Dense(
    tamil_vocab_size,
    activation="softmax",
    name="decoder_dense"
)

output = decoder_dense(decoder_output)


In [12]:
# ============================================================
# 11. BUILD MODEL
# ============================================================

model = Model(
    [encoder_input, decoder_input_layer],
    output
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    weighted_metrics=["accuracy"]
)

model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, 8)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, 6)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, 8, 64)     │     11,008 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 8)         │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, 6, 64)     │     11,456 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, 128),     │     98,816 │ encoder_embeddin… │
│                     │ (None, 128),      │            │ not_equal[0][0]   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, 6, 128),  │     98,816 │ decoder_embeddin… │
│                     │ (None, 128),      │            │ encoder_lstm[0][… │
│                     │ (None, 128)]      │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_dense       │ (None, 6, 179)    │     23,091 │ decoder_lstm[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 243,187 (949.95 KB)

 Trainable params: 243,187 (949.95 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# ============================================================
# 12. TRAIN
# ============================================================

EPOCHS = 200
BATCH_SIZE = 16

history = model.fit(
    [input_sequences, decoder_input_sequences],
    decoder_output_sequences,
    sample_weight=sample_weight,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    shuffle=True,
    verbose=1
)


Epoch 1/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1683 - loss: 4.1861  
Epoch 2/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2463 - loss: 4.1551
Epoch 3/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2463 - loss: 4.0339
Epoch 4/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2463 - loss: 3.5957
Epoch 5/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2463 - loss: 3.3942
Epoch 6/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2463 - loss: 3.2933
Epoch 7/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2463 - loss: 3.2511
Epoch 8/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2463 - loss: 3.2062
Epoch 9/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2780 - loss: 3.1570
Epoch 10/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2854 - loss: 3.1130
Epoch 11/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.2854 - loss: 3.0442
Epoch 12/200
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.2854 - loss: 2.9809

In [14]:
# ============================================================
# 13. INFERENCE MODELS
# ============================================================

encoder_model = Model(
    encoder_input,
    [state_h, state_c]
)

inf_decoder_input = Input(
    shape=(1,),
    name="inf_decoder_input"
)

inf_state_h = Input(
    shape=(LSTM_UNITS,),
    name="inf_state_h"
)

inf_state_c = Input(
    shape=(LSTM_UNITS,),
    name="inf_state_c"
)

inf_embedding = decoder_embedding_layer(
    inf_decoder_input
)

inf_output, inf_h, inf_c = decoder_lstm(
    inf_embedding,
    initial_state=[inf_state_h, inf_state_c]
)

inf_probs = decoder_dense(
    inf_output
)

decoder_model = Model(
    [inf_decoder_input, inf_state_h, inf_state_c],
    [inf_probs, inf_h, inf_c]
)

sos_id = tamil_tokenizer.word_index[SOS]
eos_id = tamil_tokenizer.word_index[EOS]

print("SOS ID:", sos_id)
print("EOS ID:", eos_id)


SOS ID: 2
EOS ID: 3


In [15]:
# ============================================================
# 14. ENGLISH → TAMIL TRANSLATION
# ============================================================

def translate(sentence, max_len=None):

    if max_len is None:
        max_len = max_output_seq_length

    # English -> token IDs
    sequence = english_tokenizer.texts_to_sequences(
        [sentence]
    )

    sequence = pad_sequences(
        sequence,
        maxlen=max_input_seq_length,
        padding="post"
    )

    # Encoder states
    h, c = encoder_model.predict(
        sequence,
        verbose=0
    )

    # Start with SOS
    current_token = np.array(
        [[sos_id]],
        dtype=np.int32
    )

    result = []

    for _ in range(max_len):

        prediction, h, c = decoder_model.predict(
            [current_token, h, c],
            verbose=0
        )

        predicted_id = int(
            np.argmax(prediction[0, -1, :])
        )

        # Stop at EOS or padding
        if predicted_id == eos_id or predicted_id == 0:
            break

        predicted_word = tamil_index_word.get(
            predicted_id,
            "<unk>"
        )

        if predicted_word not in (SOS, EOS):
            result.append(predicted_word)

        # Feed prediction into next decoder step
        current_token = np.array(
            [[predicted_id]],
            dtype=np.int32
        )

    return " ".join(result)


In [34]:
# ============================================================
# 15. TEST
# ============================================================

test_sentence = "What is your name?"

print("English:")
print(test_sentence)

print("\nTamil:")
print(translate(test_sentence))


English:
What is your name?

Tamil:
உங்கள் பெயர் என்ன?


In [25]:
# ============================================================
# 16. TEST MORE TRAINING EXAMPLES
# ============================================================

for i in [0, 1, 2, 10, 20]:

    print("\n" + "=" * 60)
    print("English :", english_sentences[i+1])
    print("Expected:", tamil_sentences[i+1])
    print("Model   :", translate(english_sentences[i]))



English : I am fine.
Expected: நான் நலமாக இருக்கிறேன்.
Model   : நான் நலமாக இருக்கிறேன்.

English : What is your name?
Expected: உங்கள் பெயர் என்ன?
Model   : உங்கள் பெயர் என்ன?

English : My name is Ravi.
Expected: என் பெயர் ரவி.
Model   : என் பெயர் ரவி.

English : I like this phone.
Expected: எனக்கு இந்த தொலைபேசி பிடிக்கும்.
Model   : எனக்கு இந்த தொலைபேசி பிடிக்கும்.

English : Please help me.
Expected: தயவுசெய்து எனக்கு உதவுங்கள்.
Model   : தயவுசெய்து எனக்கு உதவுங்கள்.


In [55]:
items=[
"The project will start tomorrow.",
"I need to finish this work today.",
"The phone has a very good camera.",
"Can you send the file tomorrow?",
"We need more time to complete the project."
]

for i in items:
    print("\n" + "=" * 60)
    print("English :", i)
    # print("Expected:", i)
    print("Model   :", translate(i))


English : The project will start tomorrow.
Model   : திட்டம் தயாராக இல்லை.

English : I need to finish this work today.
Model   : நான் அதைச் சோதிக்க சோதிக்க வேண்டும்.

English : The phone has a very good camera.
Model   : இந்த தொலைபேசியில் நல்ல கேமரா உள்ளது.

English : Can you send the file tomorrow?
Model   : இதை தரவு விளக்க

English : We need more time to complete the project.
Model   : இந்த தொலைபேசியில் நல்ல
